# Plots for FoRL

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import to_rgba

from rewarduq.utils_ext.ml import load_wandb_history
from rewarduq.utils_ext.plot import Plotter
from rewarduq.utils_ext.tools import setup_logging

plt.ioff()
setup_logging()

logger = logging.getLogger(__name__)

PATH_DATA = Path("data")
PATH_OUTPUT = Path("../../output/plots/forl")

# setup plotter

FONTSIZE_SMALL = 6
FONTSIZE_DEFAULT = 8
FONTSIZE_LARGE = 10

Plotter.setup(css_patches=["overflow_auto", "gray_background"])
Plotter.configure(
    basewidth=5.5,
    fontsize=FONTSIZE_DEFAULT,
    latex=False,
    rcparams={
        "lines.linewidth": 1,  # default: 1.5
        "axes.labelpad": 2,  # default: 4
    },
    save_dir=PATH_OUTPUT,
    save_format="pdf",
)
Plotter.configure(
    latex=True,
    latex_preamble="\n".join(
        [
            r"\usepackage[utf8]{inputenc}",
            r"\usepackage[T1]{fontenc}",
            r"\usepackage{microtype}",
            r"\usepackage{lmodern}",  # for 8-bit Latin Modern font
            r"\usepackage[sc]{mathpazo}",  # for Palatino font
            r"\usepackage{amsmath,amssymb,amsfonts,mathrsfs}",
        ]
    ),
)

In [ ]:
def load_calibration_curves(path):
    df_curves = pd.read_csv(path)
    curves = {
        name: tuple(group[key] for key in ["bin", "count", "p_true", "p_pred"])
        for name, group in df_curves.groupby("name")
    }
    return curves


def stack_histories(histories, experiments, sort_by=None):
    if experiments is not None:
        histories = {
            name: pd.concat([histories[experiment_name] for experiment_name in experiment_names])
            for name, experiment_names in experiments.items()
        }
    if sort_by is not None:
        histories = {name: df.sort_values(by=sort_by) for name, df in histories.items()}
    return histories

## Plots

In [ ]:
def plot_preferences(plot_group, path, experiments=None):
    # load data
    histories = load_wandb_history(
        path,
        {
            "train/epoch": "epoch",
            "eval/prefs/pred_mean": "pred",
            "eval/prefs/lower_mean": "lower",
            "eval/prefs/upper_mean": "upper",
        },
    )
    histories = stack_histories(histories, experiments, sort_by="epoch")

    # plot
    for name, df_history in histories.items():
        fig, ax = Plotter.create()
        ax.plot(
            df_history["epoch"],
            df_history["pred"],
        )
        ax.fill_between(
            df_history["epoch"],
            df_history["lower"],
            df_history["upper"],
            alpha=0.25,
        )
        Plotter.set(ax, xlabel="epochs", ylabel="preference probability")
        plot_group.add_plot(fig, f"rewarduq-preferences-{name}")


def plot_confidence_accuracy(plot_group, path, experiments=None):
    # load data
    histories = load_wandb_history(
        path,
        {
            "train/epoch": "epoch",
            "eval/prefs/confident_correct": "confident correct",
            "eval/prefs/ambiguous": "ambiguous",
            "eval/prefs/confident_incorrect": "confident incorrect",
        },
    )
    histories = stack_histories(histories, experiments, sort_by="epoch")

    # plot
    for name, df_history in histories.items():
        fig, ax = Plotter.create()
        ax.stackplot(
            df_history["epoch"],
            df_history["confident correct"],
            df_history["ambiguous"],
            df_history["confident incorrect"],
            labels=["confident correct", "ambiguous", "confident incorrect"],
            colors=[to_rgba(c, alpha=0.75) for c in ["tab:green", "tab:blue", "tab:red"]],
        )
        Plotter.set(ax, xlabel="epochs", legend=dict(order=[2, 1, 0]))
        plot_group.add_plot(fig, f"rewarduq-confidence_accuracy-{name}")


with Plotter.group(
    figwidth=0.4,
    grid_ncols=2,
    consistent_size=True,
    save_kw=dict(transparent=True),
    # save=True,
) as plot_group:
    plot_preferences(
        plot_group,
        PATH_DATA / "prefs.csv",
        experiments={
            "dpo_lora_rm": [
                "20250524-191100-clumsy-tern-70",
                "20250520-205242-sincere-goose-909",
                "20250522-093557-glamorous-mink-111",
                "20250522-143316-indecisive-worm-398",
            ],
        },
    )
    plot_confidence_accuracy(
        plot_group,
        PATH_DATA / "confidence_accuracy.csv",
        experiments={
            "dpo_lora_rm": [
                "20250524-191100-clumsy-tern-70",
                "20250520-205242-sincere-goose-909",
                "20250522-093557-glamorous-mink-111",
                "20250522-143316-indecisive-worm-398",
            ],
        },
    )